In [1]:
!pip install huggingface_hub

In [ ]:
from huggingface_hub import hf_hub_download

hf_hub_download(
    repo_id="facebook/sam3",
    filename="sam3.pt",
    token=""
)

sam3.pt:   0%|          | 0.00/3.45G [00:00<?, ?B/s]

'/root/.cache/huggingface/hub/models--facebook--sam3/snapshots/3c879f39826c281e95690f02c7821c4de09afae7/sam3.pt'

In [4]:
from huggingface_hub import login
login()

In [5]:
from huggingface_hub import hf_hub_download
hf_hub_download(repo_id="facebook/sam3", filename="sam3.pt")

'/root/.cache/huggingface/hub/models--facebook--sam3/snapshots/3c879f39826c281e95690f02c7821c4de09afae7/sam3.pt'

In [6]:
import os
print(os.path.exists("sam3.pt"))

False


In [8]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 31.5 MB/s eta 0:00:00


In [15]:
from google.colab import files
files.upload()
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
!kaggle datasets download -d nirmalsankalana/plantdoc-dataset
!unzip plantdoc-dataset.zip

Saving kaggle.json to kaggle.json
Dataset URL: https://www.kaggle.com/datasets/nirmalsankalana/plantdoc-dataset
License(s): CC0-1.0
 99% 884M/896M [00:09<00:00, 175MB/s]
100% 896M/896M [00:09<00:00, 102MB/s]
Archive:  plantdoc-dataset.zip
  inflating: file_renamer.py         
  inflating: folder_renamer.py       
  inflating: test/Apple_Scab_Leaf/test_Apple Scab Leaf_1.jpg  
  inflating: test/Apple_Scab_Leaf/test_Apple Scab Leaf_10.jpg  
  inflating: test/Apple_Scab_Leaf/test_Apple Scab Leaf_2.jpg  
  inflating: test/Apple_Scab_Leaf/test_Apple Scab Leaf_3.jpg  
  inflating: test/Apple_Scab_Leaf/test_Apple Scab Leaf_4.jpg  
  inflating: test/Apple_Scab_Leaf/test_Apple Scab Leaf_5.jpg  
  inflating: test/Apple_Scab_Leaf/test_Apple Scab Leaf_6.jpg  
  inflating: test/Apple_Scab_Leaf/test_Apple Scab Leaf_7.jpg  
  inflating: test/Apple_Scab_Leaf/test_Apple Scab Leaf_8.jpg  
  inflating: test/Apple_Scab_Leaf/test_Apple Scab Leaf_9.jpg  
  inflating: test/Apple_leaf/test_Apple leaf_1.jpg  
 

In [10]:
from huggingface_hub import hf_hub_download

sam_path = hf_hub_download(
    repo_id="facebook/sam3",
    filename="sam3.pt"
)

print("Model saved at:", sam_path)

Model saved at: /root/.cache/huggingface/hub/models--facebook--sam3/snapshots/3c879f39826c281e95690f02c7821c4de09afae7/sam3.pt


In [12]:
from ultralytics import SAM
import torch

DEVICE = 0 if torch.cuda.is_available() else "cpu"

sam_model = SAM(sam_path)
sam_model.to(DEVICE)

print("SAM3 loaded successfully")

SAM3 loaded successfully


In [ ]:
import os
import cv2
import numpy as np
import torch
from ultralytics import SAM

DEVICE = 0 if torch.cuda.is_available() else "cpu"

INPUT_ROOT = "train"
OUTPUT_ROOT = "/content/segmented_sam3"

selected_classes = [
    "Bell_pepper_leaf",
    "Bell_pepper_leaf_spot"
    "Potato_leaf_early_blight",
    "Potato_leaf_late_blight"
]

os.makedirs(OUTPUT_ROOT, exist_ok=True)

print("Using device:", DEVICE)
print("Loading SAM3 model...")

sam_model.to(DEVICE)

def segment_tight_crop(image):

    h, w = image.shape[:2]

    results = sam_model(image, device=DEVICE)

    if results[0].masks is None:
        return image

    masks = results[0].masks.data.cpu().numpy()

    if len(masks) == 0:
        return image

    masks = [m for m in masks if m.sum() > 0.02 * h * w]
    if len(masks) == 0:
        return image

    mask = max(masks, key=lambda x: x.sum())

    ys, xs = np.where(mask > 0)
    y1, y2 = ys.min(), ys.max()
    x1, x2 = xs.min(), xs.max()

    pad_y = int((y2 - y1) * 0.05)
    pad_x = int((x2 - x1) * 0.05)

    y1 = max(0, y1 - pad_y)
    y2 = min(h, y2 + pad_y)
    x1 = max(0, x1 - pad_x)
    x2 = min(w, x2 + pad_x)

    return image[y1:y2, x1:x2]

print("Starting SAM3 segmentation")

for cls in selected_classes:

    input_folder = os.path.join(INPUT_ROOT, cls)
    output_folder = os.path.join(OUTPUT_ROOT, cls)
    os.makedirs(output_folder, exist_ok=True)

    images = os.listdir(input_folder)
    print(f"\nProcessing {cls} ({len(images)} images)")

    for i, img_name in enumerate(images):

        img_path = os.path.join(input_folder, img_name)
        image = cv2.imread(img_path)

        if image is None:
            continue

        cropped = segment_tight_crop(image)

        save_path = os.path.join(output_folder, img_name)
        cv2.imwrite(save_path, cropped)

        if i % 10 == 0:
            print(f"{i}/{len(images)} processed")

print("\nSAM3 segmentation completed")

Using device: 0
Loading SAM3 model...
Starting SAM3 segmentation...

Processing Bell_pepper_leaf (34 images)

WARNING ⚠️ imgsz=[1024] must be multiple of max stride 14, updating to [1036]
0: 1036x1036 1 0, 1 1, 1 2, 1 3, 1 4, 1 5, 1 6, 1 7, 1 8, 1 9, 1 10, 1 11, 1 12, 1 13, 1 14, 1 15, 1 16, 1 17, 1 18, 1 19, 1 20, 1 21, 1 22, 1 23, 1 24, 1 25, 1 26, 1 27, 39273.3ms
Speed: 96.0ms preprocess, 39273.3ms inference, 1.9ms postprocess per image at shape (1, 3, 1036, 1036)
0/34 processed

WARNING ⚠️ imgsz=[1024] must be multiple of max stride 14, updating to [1036]
0: 1036x1036 1 0, 1 1, 1 2, 1 3, 1 4, 1 5, 1 6, 1 7, 1 8, 1 9, 1 10, 1 11, 1 12, 1 13, 1 14, 1 15, 1 16, 1 17, 1 18, 1 19, 1 20, 1 21, 1 22, 1 23, 1 24, 1 25, 1 26, 1 27, 1 28, 1 29, 1 30, 1 31, 1 32, 1 33, 1 34, 1 35, 1 36, 1 37, 42351.4ms
Speed: 13.2ms preprocess, 42351.4ms inference, 5.7ms postprocess per image at shape (1, 3, 1036, 1036)

WARNING ⚠️ imgsz=[1024] must be multiple of max stride 14, updating to [1036]
0: 1036x103

In [ ]:
import shutil
from google.colab import files

shutil.make_archive('segmented_sam3', 'zip', 'segmented_sam3')
files.download('segmented_sam3.zip')